In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
zip_file_path = '/content/drive/MyDrive/imgs.zip'
extraction_path = '/content/state-farm-distracted-driver-detection'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

# Unzip the file using a shell command
!unzip -q {zip_file_path} -d {extraction_path}

print(f'Contents of {zip_file_path} extracted to {extraction_path}')
print('Listing extracted files:')
!ls -F {extraction_path}

## 1. Check GPU Availability

First, let's verify that a GPU is available, which is crucial for training deep learning models efficiently.

In [ ]:
import tensorflow as tf

print("TensorFlow Version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.experimental.list_physical_devices('GPU')))

if tf.test.gpu_device_name():
    print('Default GPU Device: {}'.format(tf.test.gpu_device_name()))
else:
    print("Please install GPU version of TF")
    print("Please enable GPU runtime in Colab (Runtime -> Change runtime type -> GPU)")

## 2. Import Libraries and Define Parameters

This notebook trains a high-quality teacher model for driver-distraction classification. The validation set is driver-wise: validation images come from subjects that are not used for training.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import os

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
import seaborn as sns

from tensorflow.keras.preprocessing import image
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

try:
    from tensorflow.keras.applications import ConvNeXtTiny
except Exception:
    ConvNeXtTiny = None

from tensorflow.keras.applications import EfficientNetV2B0, EfficientNetB0

DATASET_PATH = '/content/state-farm-distracted-driver-detection/imgs/train'
DRIVER_CSV_PATH = '/content/state-farm-distracted-driver-detection/driver_imgs_list.csv'

IMG_HEIGHT = 224
IMG_WIDTH = 224
IMG_SIZE = (IMG_HEIGHT, IMG_WIDTH)
INPUT_SHAPE = (IMG_HEIGHT, IMG_WIDTH, 3)
BATCH_SIZE = 32

# This is a subject/driver-wise validation fraction, not a random image split.
DRIVER_VALIDATION_FRACTION = 0.2
VALIDATION_SPLIT = DRIVER_VALIDATION_FRACTION  # kept only for old notes/cells

TEACHER_BACKBONE = 'convnext_tiny'  # better teacher than EfficientNetB0 for this objective
EPOCHS_HEAD_TRAINING = 10
EPOCHS_FINE_TUNING = 15
HEAD_LR = 1e-3
FINE_TUNE_LR = 3e-6
FINE_TUNE_UNFREEZE_LAYERS = 30
USE_CLASS_WEIGHTS = True

SAVE_DIR = '/content/drive/MyDrive/distracted_driver_teacher_outputs'
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'best_teacher_model.keras')
CLASS_NAMES_SAVE_PATH = os.path.join(SAVE_DIR, 'class_names.json')
HISTORY_SAVE_PATH = os.path.join(SAVE_DIR, 'training_history.csv')
CM_SAVE_PATH = os.path.join(SAVE_DIR, 'confusion_matrix.png')
CLASSIFICATION_REPORT_PATH = os.path.join(SAVE_DIR, 'classification_report.csv')
SPLIT_SUMMARY_PATH = os.path.join(SAVE_DIR, 'driver_wise_split_summary.csv')
TRAIN_SPLIT_PATH = os.path.join(SAVE_DIR, 'driver_wise_train_split.csv')
VAL_SPLIT_PATH = os.path.join(SAVE_DIR, 'driver_wise_val_split.csv')

os.makedirs(SAVE_DIR, exist_ok=True)
print('Teacher backbone:', TEACHER_BACKBONE)
print('Outputs will be saved to:', SAVE_DIR)


## 3. Load Dataset with Driver-Wise Split

The State Farm dataset contains multiple images from the same driver. A random image split leaks driver identity into validation and can inflate accuracy. Here, the split is grouped by `subject`, so every validation driver is completely unseen during training.


In [ ]:
import tensorflow as tf

csv_candidates = [
    DRIVER_CSV_PATH,
    '/content/drive/MyDrive/driver_imgs_list.csv',
    '/content/drive/MyDrive/state-farm-distracted-driver-detection/driver_imgs_list.csv',
]
metadata_path = next((path for path in csv_candidates if os.path.exists(path)), None)
if metadata_path is None:
    raise FileNotFoundError(
        'driver_imgs_list.csv was not found. Upload it beside the dataset or to MyDrive.'
    )

metadata = pd.read_csv(metadata_path)
required_columns = {'subject', 'classname', 'img'}
missing = required_columns - set(metadata.columns)
if missing:
    raise ValueError(f'Missing required columns from driver CSV: {missing}')

metadata['label'] = metadata['classname']
metadata['path'] = metadata.apply(
    lambda row: os.path.join(DATASET_PATH, row['classname'], row['img']),
    axis=1
)
metadata = metadata[metadata['path'].map(os.path.exists)].reset_index(drop=True)
if metadata.empty:
    raise RuntimeError('No images from driver_imgs_list.csv were found under DATASET_PATH.')

class_names = sorted(metadata['label'].unique())
class_to_id = {name: idx for idx, name in enumerate(class_names)}
id_to_class = {idx: name for name, idx in class_to_id.items()}
NUM_CLASSES = len(class_names)
metadata['label_id'] = metadata['label'].map(class_to_id).astype('int32')

print(f'Loaded {len(metadata)} images from {metadata_path}')
print(f'Classes ({NUM_CLASSES}): {class_names}')
print(f'Drivers: {metadata["subject"].nunique()}')

# Try many grouped splits and choose the one with the most balanced validation classes.
def split_score(candidate_val_df):
    counts = candidate_val_df['label'].value_counts().reindex(class_names, fill_value=0)
    expected = len(candidate_val_df) / NUM_CLASSES
    class_balance_error = float(np.abs(counts - expected).sum() / max(len(candidate_val_df), 1))
    missing_classes_penalty = float((counts == 0).sum())
    return class_balance_error + missing_classes_penalty

best_split = None
best_score = float('inf')
for seed in range(300):
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=DRIVER_VALIDATION_FRACTION,
        random_state=seed
    )
    train_idx, val_idx = next(splitter.split(metadata, groups=metadata['subject']))
    candidate_train = metadata.iloc[train_idx].reset_index(drop=True)
    candidate_val = metadata.iloc[val_idx].reset_index(drop=True)
    score = split_score(candidate_val)
    if score < best_score:
        best_score = score
        best_split = (seed, candidate_train, candidate_val)

split_seed, train_df, val_df = best_split
train_subjects = set(train_df['subject'].unique())
val_subjects = set(val_df['subject'].unique())
overlap = train_subjects.intersection(val_subjects)
if overlap:
    raise RuntimeError(f'Driver leakage detected: {sorted(overlap)}')

print(f'Selected driver-wise split seed: {split_seed}')
print(f'Train images: {len(train_df)} from {len(train_subjects)} drivers')
print(f'Validation images: {len(val_df)} from {len(val_subjects)} drivers')
print('Validation drivers:', sorted(val_subjects))
print('Driver overlap:', len(overlap))

split_summary = pd.DataFrame({
    'train_count': train_df['label'].value_counts().reindex(class_names, fill_value=0),
    'val_count': val_df['label'].value_counts().reindex(class_names, fill_value=0),
})
split_summary['val_fraction'] = split_summary['val_count'] / (split_summary['train_count'] + split_summary['val_count'])
display(split_summary)

print('\nTrain subject x class counts:')
display(pd.crosstab(train_df['subject'], train_df['label']).reindex(columns=class_names, fill_value=0))
print('\nValidation subject x class counts:')
display(pd.crosstab(val_df['subject'], val_df['label']).reindex(columns=class_names, fill_value=0))

train_df.to_csv(TRAIN_SPLIT_PATH, index=False)
val_df.to_csv(VAL_SPLIT_PATH, index=False)
split_summary.to_csv(SPLIT_SUMMARY_PATH)
with open(CLASS_NAMES_SAVE_PATH, 'w') as f:
    json.dump(class_names, f, indent=2)

AUTOTUNE = tf.data.AUTOTUNE

def load_image(path, label_id):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    return img, label_id

def make_dataset(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((frame['path'].values, frame['label_id'].values))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(frame), seed=42, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df, shuffle=False)

if USE_CLASS_WEIGHTS:
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.arange(NUM_CLASSES),
        y=train_df['label_id'].values
    )
    class_weight_dict = {idx: float(weight) for idx, weight in enumerate(weights)}
else:
    class_weight_dict = None

print('\nClass weights:', class_weight_dict)
print('Split files saved to Drive.')



## 4. Data Augmentation

The teacher sees light geometric and color variation. The augmentation is intentionally modest because driver posture and hand position are important visual cues.


In [ ]:
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.035),
    tf.keras.layers.RandomZoom(0.08),
    tf.keras.layers.RandomContrast(0.18),
    tf.keras.layers.RandomBrightness(0.08),
], name='teacher_augmentation')


## 5. Build Teacher Model using Transfer Learning

For the teacher, use ConvNeXtTiny when available. It is a stronger modern convolutional backbone than EfficientNetB0 for learning posture and object cues. If the Colab TensorFlow build does not include ConvNeXtTiny, the notebook falls back to EfficientNetV2B0 and then EfficientNetB0.


In [ ]:
def make_backbone():
    if TEACHER_BACKBONE == 'convnext_tiny' and ConvNeXtTiny is not None:
        try:
            base = ConvNeXtTiny(
                include_top=False,
                weights='imagenet',
                input_shape=INPUT_SHAPE,
                include_preprocessing=True,
            )
            return base, None, 'ConvNeXtTiny'
        except TypeError:
            base = ConvNeXtTiny(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE)
            return base, tf.keras.applications.convnext.preprocess_input, 'ConvNeXtTiny'

    try:
        base = EfficientNetV2B0(
            include_top=False,
            weights='imagenet',
            input_shape=INPUT_SHAPE,
            include_preprocessing=True,
        )
        return base, None, 'EfficientNetV2B0'
    except TypeError:
        base = EfficientNetB0(include_top=False, weights='imagenet', input_shape=INPUT_SHAPE)
        return base, tf.keras.applications.efficientnet.preprocess_input, 'EfficientNetB0'


def build_teacher_model(num_classes):
    global base_model, preprocess_input, backbone_name
    base_model, preprocess_input, backbone_name = make_backbone()
    base_model.trainable = False

    inputs = tf.keras.Input(shape=INPUT_SHAPE, name='teacher_input')
    x = augmentation(inputs)
    if preprocess_input is not None:
        x = tf.keras.layers.Lambda(preprocess_input, name='backbone_preprocess')(x)
    x = base_model(x, training=False)
    x = GlobalAveragePooling2D(name='global_pool')(x)
    x = Dropout(0.45, name='teacher_dropout')(x)
    outputs = Dense(num_classes, activation='softmax', name='class_probs')(x)
    model = Model(inputs, outputs, name=f'{backbone_name}_driver_teacher')
    return model

model = build_teacher_model(NUM_CLASSES)
model.summary()
print('Backbone selected:', backbone_name)


## 6. Callbacks

The checkpoint tracks validation accuracy on unseen drivers. Early stopping also uses validation accuracy because this project cares about driver-generalized classification, not memorizing the training drivers.


In [ ]:
checkpoint_callback = ModelCheckpoint(
    filepath=MODEL_SAVE_PATH,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_accuracy',
    mode='max',
    patience=6,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    mode='min',
    factor=0.3,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callbacks = [checkpoint_callback, early_stopping, reduce_lr]


## 7. Train Model - Phase 1: Classification Head

The ImageNet backbone is frozen first. Only the new classifier head learns the distracted-driver classes.


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=HEAD_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('\n--- Training the classification head with frozen backbone ---')
history_head = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD_TRAINING,
    callbacks=callbacks,
    class_weight=class_weight_dict
)



## 8. Train Model - Phase 2: Fine-Tuning

Only the last part of the backbone is unfrozen, using a very small learning rate. Normalization layers stay frozen to avoid destabilizing the pretrained features.


In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-FINE_TUNE_UNFREEZE_LAYERS]:
    layer.trainable = False

for layer in base_model.layers:
    if isinstance(layer, (tf.keras.layers.BatchNormalization, tf.keras.layers.LayerNormalization)):
        layer.trainable = False

trainable_layers = sum(1 for layer in base_model.layers if layer.trainable)
print(f'Trainable backbone layers: {trainable_layers} / {len(base_model.layers)}')

model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('\n--- Fine-tuning last backbone layers on driver-wise split ---')
history_fine_tune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD_TRAINING + EPOCHS_FINE_TUNING,
    initial_epoch=len(history_head.epoch),
    callbacks=callbacks,
    class_weight=class_weight_dict
)



## 9. Evaluate Model and Save Results

Evaluation is performed on the held-out validation drivers only. This is the number to report for a research comparison because it checks generalization to unseen drivers.


In [ ]:
# Load best validation-accuracy checkpoint.
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH, safe_mode=False, compile=False)
best_model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

val_loss, val_accuracy = best_model.evaluate(val_ds, verbose=1)
print(f'Best driver-wise validation accuracy: {val_accuracy:.4f}')
print(f'Best driver-wise validation loss: {val_loss:.4f}')

history_frames = []
for phase_name, history_obj in [('head', globals().get('history_head')), ('fine_tune', globals().get('history_fine_tune'))]:
    if history_obj is None:
        continue
    phase_history = pd.DataFrame(history_obj.history)
    phase_history['phase'] = phase_name
    phase_history['epoch'] = np.arange(1, len(phase_history) + 1)
    history_frames.append(phase_history)

if history_frames:
    history_df = pd.concat(history_frames, ignore_index=True)
    history_df.to_csv(HISTORY_SAVE_PATH, index=False)
    display(history_df.tail())

    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history_df.index + 1, history_df['accuracy'], label='Train Accuracy')
    plt.plot(history_df.index + 1, history_df['val_accuracy'], label='Validation Accuracy')
    plt.title('Teacher Accuracy - Driver-Wise Split')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history_df.index + 1, history_df['loss'], label='Train Loss')
    plt.plot(history_df.index + 1, history_df['val_loss'], label='Validation Loss')
    plt.title('Teacher Loss - Driver-Wise Split')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.tight_layout()
    plt.show()

y_true = []
y_pred = []
for batch_images, batch_labels in val_ds:
    probs = best_model.predict(batch_images, verbose=0)
    y_true.extend(batch_labels.numpy().tolist())
    y_pred.extend(np.argmax(probs, axis=1).tolist())

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Unseen Validation Drivers')
plt.tight_layout()
plt.savefig(CM_SAVE_PATH, dpi=200)
plt.show()
print(f'Confusion matrix plot saved to {CM_SAVE_PATH}')

report_dict = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)
report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(CLASSIFICATION_REPORT_PATH)
print('\n--- Classification Report: Held-Out Drivers ---')
print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))
print(f'Classification report saved to {CLASSIFICATION_REPORT_PATH}')



## 10. Test Function for Single Image Prediction

Use this cell after training to test any image. The prediction uses the saved best teacher checkpoint.


In [ ]:
def predict_one_image(img_path, model_path=MODEL_SAVE_PATH, top_k=5):
    if not os.path.exists(img_path):
        raise FileNotFoundError(img_path)

    inference_model = tf.keras.models.load_model(model_path, safe_mode=False, compile=False)
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.expand_dims(img, axis=0)

    probs = inference_model.predict(img, verbose=0)[0]
    order = np.argsort(probs)[::-1][:top_k]

    print(f'\n--- Prediction for {os.path.basename(img_path)} ---')
    print(f'Predicted Class: {class_names[int(order[0])]}')
    print(f'Confidence: {probs[int(order[0])]:.4f}')
    print(f'Top {top_k} Class Probabilities:')
    for idx in order:
        print(f'  {class_names[int(idx)]}: {probs[int(idx)]:.4f}')
    return probs

# Example:
# predict_one_image('/content/img_9992.jpg')



## 11. Explanation: Teacher-Student Distillation for ESP32

The teacher model is intentionally larger and more accurate. It is trained on 224 x 224 RGB images with a driver-wise validation split. After the teacher is good, a small ESP32 student can learn from it by matching both the true labels and the teacher probability distribution.

In this project, distillation means:

1. Train the teacher on the original images.
2. Use the teacher to produce soft class probabilities for training images.
3. Cache those teacher probabilities so the laptop does not need to run the teacher every student epoch.
4. Train a tiny student model using a mix of hard-label loss and distillation loss.
5. Convert the student to INT8 TFLite for ESP32/Wokwi benchmarking.

The teacher accuracy should be reported with the driver-wise validation result, because that measures whether the model generalizes to unseen people rather than remembering drivers from the training set.
